# 04 - SHAP & Feature Importance (CPU, skew alignment)
Explain models, align SHAP with statistical skew/outliers, export artifacts to artifacts/shap.

## 1) Environment Setup
CPU-only; uses pandas/numpy/sklearn/matplotlib; tries shap if installed. No Colab.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

plt.style.use("dark_background")
sns.set_theme(style="darkgrid")

ROOT = Path(".").resolve()
DATA_PATH = Path(os.getenv("FRAUD_DATA_PATH", ROOT / "data" / "processed" / "transactions.parquet"))
ARTIFACT_DIR = ROOT / "artifacts" / "shap"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RNG_SEED = 42
np.random.seed(RNG_SEED)

print(f"Using data path: {DATA_PATH}")
print(f"Artifacts → {ARTIFACT_DIR}")

## 2) Load + Feature Engineering
Reuse engineered features and outlier flags; synthesize if data missing.

In [ ]:
def synthesize_transactions(n_rows: int = 50000, fraud_ratio: float = 0.01, rng_seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(rng_seed)
    labels = rng.choice([0, 1], size=n_rows, p=[1 - fraud_ratio, fraud_ratio])
    amount = rng.gamma(shape=2.0, scale=200.0, size=n_rows)
    oldbalanceOrg = rng.normal(loc=5000, scale=1500, size=n_rows)
    newbalanceOrig = oldbalanceOrg - amount * rng.uniform(0.8, 1.0, size=n_rows)
    oldbalanceDest = rng.normal(loc=2000, scale=1000, size=n_rows)
    newbalanceDest = oldbalanceDest + amount * rng.uniform(0.7, 1.0, size=n_rows)
    tx_type = rng.choice(["PAYMENT", "TRANSFER", "CASH_OUT", "DEBIT"], size=n_rows)
    return pd.DataFrame({
        "amount": amount,
        "oldbalanceOrg": oldbalanceOrg,
        "newbalanceOrig": newbalanceOrig,
        "oldbalanceDest": oldbalanceDest,
        "newbalanceDest": newbalanceDest,
        "type": tx_type,
        "is_fraud": labels,
    })


def load_dataset(path: Path) -> pd.DataFrame:
    if path.exists():
        if path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
        else:
            df = pd.read_csv(path)
        print(f"Loaded dataset from {path} with shape {df.shape}")
        return df
    print(f"Data path not found: {path}. Synthesizing sample dataset (~1% fraud).")
    return synthesize_transactions()


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["amount_log"] = np.log1p(df["amount"].clip(lower=0))
    df["balance_delta_org"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
    df["balance_delta_dest"] = df["newbalanceDest"] - df["oldbalanceDest"]
    for col in ["amount", "balance_delta_org", "balance_delta_dest"]:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        df[f"outlier_flag_{col}"] = ((df[col] < lower) | (df[col] > upper)).astype(int)
    return df


df_raw = load_dataset(DATA_PATH)
df = add_features(df_raw)

cat_cols = ["type"]
num_cols = [c for c in df.columns if c not in cat_cols + ["is_fraud"]]

numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
categorical_transformer = Pipeline(steps=[("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RNG_SEED, stratify=y
)

print("Train/Test shapes:", X_train.shape, X_test.shape)

## 3) Train compact models for SHAP
Use Logistic Regression (global weights) and Random Forest; both CPU-friendly.

In [ ]:
lr_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(max_iter=400, class_weight="balanced", n_jobs=-1, C=0.6)),
])
rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=160,
        max_depth=14,
        min_samples_leaf=2,
        n_jobs=-1,
        class_weight="balanced_subsample",
        random_state=RNG_SEED,
    )),
])

models = {"log_reg": lr_model, "rf": rf_model}

trained_models = {}
for name, m in models.items():
    print(f"Training {name} ...")
    m.fit(X_train, y_train)
    trained_models[name] = m
print("Training done.")

## 4) SHAP vs Statistical Skew Alignment
Compute SHAP values (TreeExplainer for RF if available, LinearExplainer for LR). Compare with outlier/skew signals.

In [ ]:
if not HAS_SHAP:
    print("shap not installed; install to compute SHAP values: pip install shap")
else:
    # Use a sample to keep CPU reasonable
    sample_size = min(5000, len(X_test))
    X_sample = X_test.sample(sample_size, random_state=RNG_SEED)

    # Transform once for LR background (dense)
    lr_pre = trained_models["log_reg"].named_steps["preprocess"]
    X_sample_enc = lr_pre.transform(X_sample)

    # SHAP for Logistic Regression
    explainer_lr = shap.LinearExplainer(trained_models["log_reg"].named_steps["clf"], X_sample_enc)
    shap_values_lr = explainer_lr.shap_values(X_sample_enc)
    shap.summary_plot(shap_values_lr, X_sample_enc, show=False)
    lr_summary_path = ARTIFACT_DIR / "shap_log_reg_summary.png"
    plt.tight_layout()
    plt.savefig(lr_summary_path)
    plt.close()
    print(f"Saved LR SHAP summary to {lr_summary_path}")

    # SHAP for Random Forest (tree explainer on transformed dense array)
    rf_pre = trained_models["rf"].named_steps["preprocess"]
    X_sample_enc_rf = rf_pre.transform(X_sample)
    explainer_rf = shap.TreeExplainer(trained_models["rf"].named_steps["clf"], feature_perturbation="interventional")
    shap_values_rf = explainer_rf.shap_values(X_sample_enc_rf)
    shap.summary_plot(shap_values_rf[1], X_sample_enc_rf, show=False)
    rf_summary_path = ARTIFACT_DIR / "shap_rf_summary.png"
    plt.tight_layout()
    plt.savefig(rf_summary_path)
    plt.close()
    print(f"Saved RF SHAP summary to {rf_summary_path}")

## 5) Skew/Outlier vs SHAP Consistency
Compare SHAP magnitude ranks vs statistical skew (variance, IQR outliers).

In [ ]:
# Statistical skew/outlier scores
stats = df[num_cols].agg(['mean', 'var']).T
stats['iqr'] = df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25)
stats_sorted = stats.sort_values(by='var', ascending=False)

# If SHAP was computed, use mean(|SHAP|) to compare top drivers
if HAS_SHAP:
    # For LR, shap_values_lr shape (n_samples, n_features)
    mean_abs_lr = np.abs(shap_values_lr).mean(axis=0)
    shap_rank_lr = pd.Series(mean_abs_lr, index=range(len(mean_abs_lr)))

    # For RF, shap_values_rf is list [class0, class1]; take class1
    mean_abs_rf = np.abs(shap_values_rf[1]).mean(axis=0)
    shap_rank_rf = pd.Series(mean_abs_rf, index=range(len(mean_abs_rf)))

    # NOTE: feature names after preprocessing are expanded; for quick view we focus on top raw numeric drivers
    # Simplified view: map original numeric cols to SHAP indices from the numeric transformer portion only

print("Top variance features:")
display(stats_sorted.head(10))


## 6) Export Artifacts
Persist SHAP summaries and skew stats.

In [ ]:
stats_path = ARTIFACT_DIR / "skew_stats.csv"
stats_sorted.to_csv(stats_path)
print(f"Saved skew stats to {stats_path}")

# Note: SHAP plots saved earlier; add placeholder if shap missing
if not HAS_SHAP:
    with open(ARTIFACT_DIR / "README.txt", "w") as f:
        f.write("Install shap to compute SHAP values: pip install shap\n")
    print("SHAP not installed; wrote reminder to artifacts/shap/README.txt")